<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/modelo_econometrico_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programa para execução de 10 modelos econométricos

>Trata-se de notebook para execução de 10 modelos econométricos em cima do arquivo **painel_uf_2019_2024_austeridade.csv**.



  ---
  **Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026
  
  ---

# Modelos Econométricos de Efeitos Fixos - Dados em Painel


## Panorama  da Gestão Fiscal Responsável nos Estados Brasileiros (mandatos: 2019 até 2022 e 2023-2024)
---
**Detalhamento:** Dez modelos econométricos de efeitos fixos para identificar circunstâncias determinantes de um gestor público do Executivo Estadual que rege sua administração fiscal de forma responsável , baseando-se nos critérios da Lei de Responsabilidade Fiscal (LRF).

**Variável Dependente/Explicativa (Y):** **Gestao_Fiscal_Responsável** (binária: 1 = gestão responsável conforme LRF ou  0 = gestão "esbanjadora" com gastos)

**Critério de Classificação:** A gestão é considerada responsável quando a Despesa Total com Pessoal (DTP) representa até 49% da Receita Corrente Líquida Ajustada (RCLA), conforme limite estabelecido no Art. 20 da LRF.

In [ ]:
# ======================================================================
# IMPORTAÇÃO DE BIBLIOTECAS
# ======================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Bibliotecas para econometria
import statsmodels.api as sm
from scipy import stats

# Bibliotecas para visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Limites de Visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


In [ ]:
# ======================================================================
# CARREGAMENTO E PREPARAÇÃO DOS DADOS
# ======================================================================
# Carregar o dataset
df = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/2026-mestrado-pep/refs/heads/main/data/painel_uf_2019_2024_completo.csv')

print(df.describe())

               ano           dcl           pop       pib_prc           rcl  \
count   162.000000  1.620000e+02  1.620000e+02  1.620000e+02  1.620000e+02   
mean   2021.500000  3.055441e+10  7.807304e+06  2.779870e+11  3.403981e+10   
std       1.713121  6.539933e+10  9.145193e+06  5.167868e+11  4.024348e+10   
min    2019.000000 -7.186553e+09  5.765680e+05  0.000000e+00  4.200526e+09   
25%    2020.000000  1.819053e+09  2.839188e+06  2.672352e+10  1.231633e+10   
50%    2021.500000  5.301876e+09  4.059905e+06  1.121001e+11  2.209862e+10   
75%    2023.000000  1.323936e+10  9.432366e+06  2.734624e+11  3.816458e+10   
max    2024.000000  3.176556e+11  4.664913e+07  3.444814e+12  2.513670e+11   

               rcla           dtp  percentual_dtp_rcla  austeridade  \
count  1.620000e+02  1.620000e+02           162.000000   162.000000   
mean   3.397402e+10  1.464260e+10            43.303951     0.370370   
std    4.018673e+10  1.718791e+10             5.149625     0.484401   
min    4.2005

## 1. Principais colunas do arquivo base do Problema


### 1.1 Variável Dependente (Y): Gestao_Fiscal_Responsável

A variável dependente binária **Gestao_Fiscal_Responsável** é fictícia e criada com base nos critérios estabelecidos pela Lei de Responsabilidade Fiscal (LRF):

- **Gestao_Fiscal_Responsável = 1**: Quando a Despesa Total com Pessoal (DTP) representa até 49% da Receita Corrente Líquida Ajustada (RCLA), conforme Art. 20 da LRF
- **Gestao_Fiscal_Responsável = 0**: Quando a DTP supera 49% da RCLA, indicando descumprimento do limite legal.



>IMPORTANTE: Observa-se que esta Y é altamente relacionada a três colunas do dataset: dtp, rcla e percentual_dtp_rcla. **Basicamente não faz sentido essas 3 colunas continuarem no dataset, portanto serão eliminadas do modelo pois já explicam diretamente o cálculo de Y**


  


### 1.2 Variáveis Independentes

1. **Dívida Consolidada Líquida** (*dcl*) — valor absoluto da dívida menos disponibilidades (recursos em curto prazo disponíveis em caixa/estoque do governo). Proveniente do Anexo 2 do RGF.
1. **População do Estado no Ano** ( *pop* ) - trata-se da população que foi informada pelo Ente no RGF daquele ano.
1. **PIB de Preços Correntes** ( *pib_prec_corr* ) - trata-se de PIB de todas as receitas e riquezas apuradas pelo ente naquele ano. Dado extraído do SIDRA/IBGE, por isso o dado do ano 2024 está ainda indisponível até o presente momento da pesquisa. Na falta do dado do ano de 2024, atribuído valor 0.
1. **Receita Corrente Líquida** ( rcl ) - trata-se de valor absoluto da Receita Corrente Líquida do ente.
1. **Receita Corrente Líquida Ajustada** ( rcla ) - trata-se do valor de  Receita Corrente Líquida reduzido do montante de Transferências Obrigatórias da União Relativas às Emendas Individuais (art. 166-A, §1º, da CF).
1. **Despesa Total com Pessoal** (dtp) - Somatório de gastos com pessoal.
1. **Despesa Total com Pessoal como % da RCLA** (dtp_perc_rcla) - Razão entre a despesa total com pessoal (dtp) em relação ao valor da Receita Corrente Líquida Ajustada (rcla) em percentual. Proxy de rigidez fiscal embasado em norma (art. 20 e 22, da LRF), que estabelece governadores não podem comprometer mais de 49%
1. **Austeridade** ( austeridade ) é uma variável exógena ao contexto de gestão fiscal e sim trazida do resultado do pleito eleitoral de 2018 e de 2022. Ela  será 1 se o governador eleito for dos seguintes partidos abaixo e 0 nos demais casos:
    - NOVO
    - PL
    - PSDB
    - DEM
    - UNIÂO
    - PSD
    - REPUBLICANOS
1. **Mesmo Partido** (dummy mesmopartido) é uma variável que compara se o atual vencedor é do mesmo partido do governador anterior. Na prática, ela será valor 1 se sim, casos em que ele se reelegeu ou elegeu seu sucessor. Se for candidato vencedor de outro partido será 0.
1. **Reeleito** (dummy reeleito) é uma variável que comparará o mandato anterior e se o candidato é o mesmo que já era governador antes, o que denota que já havia experiência no cargo. Se sim , ela terá valor 1. Caso contrário, será valor 0.



### 1.3 Distribuição da variável Y (Gestão Fiscal Responsável) por ano e por estado.

In [ ]:
# =============================================================================
# CRIAÇÃO DA VARIÁVEL DEPENDENTE
# =============================================================================


# Verificar a distribuição de Y
print("=" * 80)
print("DISTRIBUIÇÃO DA VARIÁVEL DEPENDENTE")
print("=" * 80)
print("\n Porcentagem de gestão responsável = 1, a cada ano:")
print((df.groupby('ano')['gestao_fiscal_responsavel'].mean() * 100))

print("\n Divisão Por ano (valor absoluto):")
print(df.groupby('ano')['gestao_fiscal_responsavel'].value_counts().unstack(fill_value=0))



DISTRIBUIÇÃO DA VARIÁVEL DEPENDENTE

 Porcentagem de gestão responsável = 1, a cada ano:
ano
2019    77.777778
2020    85.185185
2021    92.592593
2022    96.296296
2023    88.888889
2024    96.296296
Name: gestao_fiscal_responsavel, dtype: float64

 Divisão Por ano (valor absoluto):
gestao_fiscal_responsavel  0   1
ano                             
2019                       6  21
2020                       4  23
2021                       2  25
2022                       1  26
2023                       3  24
2024                       1  26


## 2. Transformações de Variáveis

**Sugere-se dois modelos econométricos LOGIT em que as seguintes tranformações serão necessárias na base ou em variáveis**

### 2.1 Eliminação da base de PIB a preços correntes zerados.
- Ocorre no ano de 2024 que este indicador não está disponível, portanto ou trabalha-se com PIB defasado a ano anterior ou elimina-se linhas em que o PIB a preços correntes é zero. A segundo opção é menos danosa de efeitos.


### 2.2. Transformações Logarítmicas:
Utilizadas para lidar com valores muito grandes e reduzir a heterocedasticidade, no caso duas variáveis estão em escala muito maior que as demais e serão aplicadas transformação logarítmica nelas:
- **log_rcl** = log(rcl)
- **log_pop** = log(pop)


### 2.3. Transformação do Seno Hiperbólico Inverso
Utilizadas para lidar com dado negativo que ocorre em Dívida Consolidada Líquida. Na realidade não trata-se de uma "leitura suja" ou incongruência e sim um fator benéfico no qual um ente inda terá uma margem para se endividar ,ou seja, é um indicador de "saúde" das contas públicaws
- **arcsinh_dcl** = arcsinh(dcl)


### 2.4 Tamanho do Ente em cada ano
Trata-se de uma proxy econômica que tenta medir o tamanho financeiro do ente em termos de sua população e sua riqueza total, através do cálculo de PIB a preços correntes. Como são variáveis em escala que exorbitam as demais no modelo será aplicada uma razão e posterior log nelas.

- **log_PIB_pop** = log(PIP_pc / pop)


### 2.5 Variáveis de Controle Não Fiscais mantidas
- **austeridade** (variável exógena da dimensão fiscal)
- **reeleito** (variável exógena da dimensão fiscal, uma dummy que mede representa experiência do gestor)
- **mesmopartido** (variável exógena, uma dummy que aponta para o passado , se o grupo político se perpetuou via mesmo governador ou novo partidário eleito)


### 2.6 Eliminação da base de colunas que já definem Y.
- Eliminados percentual_dtp_rcla, dtp e rcla.

In [ ]:
# =============================================================================
# TRANSFORMAÇÕES DE VARIÁVEIS
# =============================================================================

# remover observações com PIB faltante ou zero
df = df[df["pib_prc"] > 0]
df = df[df["ano"] != 2024]

# log população
df["log_pop"] = np.log(df["pop"])
# log receita corrente liquida
df["log_rcl"] = np.log(df["rcl"])

# transformação sno hiperbolico inverso para dívida
df["arcsinh_dcl"] = np.arcsinh(df["dcl"])

# remover colunas desnecessárias
df = df.drop(columns=["percentual_dtp_rcla","dtp","rcla"], errors="ignore")



# criar PIB per capita, como razão entre PIP a preços correntes e População
df["pib_pc"] = df["pib_prc"] / df["pop"]

# aplicar log do PIB per capita
df["log_pib_pc"] = np.log(df["pib_pc"])


df.head(200)


,uf,ano,dcl,pop,pib_prc,rcl,austeridade,mesmopartido,reeleito,gestao_fiscal_responsavel,log_pop,log_rcl,arcsinh_dcl,pib_pc,log_pib_pc
0,AC,2019,3.116892e+09,869265,1.563002e+10,5.357456e+09,0,0,0,0,13.675403,22.401755,22.553249,17980.727396,9.797056
1,AC,2020,3.337030e+09,881935,1.647637e+10,5.702871e+09,0,0,0,0,13.689874,22.464236,22.621494,18682.069540,9.835319
2,AC,2021,2.847799e+09,894470,2.137444e+10,6.690646e+09,0,0,0,0,13.703987,22.623976,22.462959,23896.206692,10.081475
3,AC,2022,2.505322e+09,906876,2.367614e+10,7.994707e+09,0,0,0,1,13.717761,22.802046,22.334830,26107.357566,10.169972
4,AC,2023,2.051524e+09,906876,2.629132e+10,8.573004e+09,0,1,1,1,13.717761,22.871884,22.134996,28991.086984,10.274744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,TO,2019,4.564454e+09,1555229,3.935594e+10,7.365653e+09,0,1,1,1,14.257133,22.720094,22.934712,25305.560146,10.138779
157,TO,2020,3.246271e+09,1572866,4.364980e+10,8.198916e+09,0,1,1,1,14.268410,22.827268,22.593920,27751.762070,10.231055
158,TO,2021,1.970009e+09,1590248,5.178076e+10,1.005317e+10,0,1,1,1,14.279401,23.031154,22.094451,32561.439474,10.390884
159,TO,2022,9.196529e+08,1607363,5.820880e+10,1.210600e+10,0,1,1,1,14.290106,23.216967,21.332654,36213.845286,10.497197


## Especificação e resultados de modelos

### 3.1 Modelo 1 : Logit Efeito Fixo

> Desconsidera o formato em Painel da base, portanto cada observação é uma linha independente, por isso é esperado o problema abaixo.

In [ ]:
# =============================================================================
# IMPLEMENTAÇÃO DO MODELO DE EFEITO FIXO (LOGIT)
# =============================================================================
import statsmodels.formula.api as smf
df_model1 = df.copy()


# garantir que UF e ano são categóricos
df_model1["uf"] = df_model1["uf"].astype("category")
df_model1["ano"] = df_model1["ano"].astype("category")

# modelo logit com efeitos fixos
modelo1 = smf.logit(
    "gestao_fiscal_responsavel ~ austeridade + mesmopartido + reeleito + log_pib_pc + log_pop + arcsinh_dcl + C(uf) + C(ano)",
    data=df_model1
).fit(method='bfgs', maxiter=1000)


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 164
         Function evaluations: 175
         Gradient evaluations: 175


**IMPORTANTE**: Aumento para 1000 iterações e mesmo assim não convergiu modelo!

In [ ]:
modelo1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               Logit Regression Results                              
=====================================================================================
Dep. Variable:     gestao_fiscal_responsavel   No. Observations:                  135
Model:                                 Logit   Df Residuals:                       98
Method:                                  MLE   Df Model:                           36
Date:                       Mon, 09 Mar 2026   Pseudo R-squ.:                   1.000
Time:                               06:17:46   Log-Likelihood:            -5.8295e-05
converged:                              True   LL-Null:                       -49.135
Covariance Type:                   nonrobust   LLR p-value:                 1.100e-07
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept       -1.99e+04        nan        nan        nan         nan         nan
C(uf)[T.AL]     1.321e+04        nan        nan        nan         nan         nan
C(uf)[T.AM]     1.384e+04        nan        nan        nan         nan         nan
C(uf)[T.AP]     3.016e+04        nan        nan        nan         nan         nan
C(uf)[T.BA]     1.027e+04        nan        nan        nan         nan         nan
C(uf)[T.CE]     1.048e+04        nan        nan        nan         nan         nan
C(uf)[T.DF]     3.258e+04        nan        nan        nan         nan         nan
C(uf)[T.ES]      2.21e+04        nan        nan        nan         nan         nan
C(uf)[T.GO]    -2.226e+04        nan        nan        nan         nan         nan
C(uf)[T.MA]     1.836e+04        nan        nan        nan         nan         nan
C(uf)[T.MG]    -3.388e+04        nan        nan        nan         nan         nan
C(uf)[T.MS]     8787.5644        nan        nan        nan         nan         nan
C(uf)[T.MT]    -2.418e+04        nan        nan        nan         nan         nan
C(uf)[T.PA]     1.648e+04        nan        nan        nan         nan         nan
C(uf)[T.PB]     1963.9073        nan        nan        nan         nan         nan
C(uf)[T.PE]     1.208e+04        nan        nan        nan         nan         nan
C(uf)[T.PI]     1.259e+04        nan        nan        nan         nan         nan
C(uf)[T.PR]     1.066e+04        nan        nan        nan         nan         nan
C(uf)[T.RJ]     2.894e+04        nan        nan        nan         nan         nan
C(uf)[T.RN]    -7.826e+04        nan        nan        nan         nan         nan
C(uf)[T.RO]     2.602e+04        nan        nan        nan         nan         nan
C(uf)[T.RR]     1.177e+04        nan        nan        nan         nan         nan
C(uf)[T.RS]     1.749e+04        nan        nan        nan         nan         nan
C(uf)[T.SC]     2.466e+04        nan        nan        nan         nan         nan
C(uf)[T.SE]     1.558e+04        nan        nan        nan         nan         nan
C(uf)[T.SP]     1.247e+04        nan        nan        nan         nan         nan
C(uf)[T.TO]     1.534e+04        nan        nan        nan         nan         nan
C(ano)[T.2020]   -86.7776        nan        nan        nan         nan         nan
C(ano)[T.2021]   1.23e+04        nan        nan        nan         nan         nan
C(ano)[T.2022]  3.888e+04        nan        nan        nan         nan         nan
C(ano)[T.2023]  5080.0905        nan        nan        nan         nan         nan
austeridade     3.641e+04        nan        nan        nan         nan         nan
mesmopartido    1.291e+04        nan        nan        nan         nan         nan
reeleito       -6015.7913        nan        nan        nan         nan         nan
log_pib_pc      2331.4874        nan        nan        nan         nan         nan
log_pop        -1156.939

### 3.2 Modelo 2 - Mundlak-Wooldridge

In [ ]:
# Criação de replica
df_model2 = df.copy()

# Médias intra-UF (controle de Mundlak — substitui C(uf) corretamente)
for v in ["arcsinh_dcl", "log_pop", "log_pib_pc"]:
    df_model2[f"{v}_mean"] = df_model2.groupby("uf")[v].transform("mean")

# Dummies de ANO (sem separação perfeita — viável)
modelo2 = smf.logit(
    """gestao_fiscal_responsavel ~
       austeridade + mesmopartido + reeleito +
       log_pib_pc + log_pop + arcsinh_dcl +
       arcsinh_dcl_mean + log_pop_mean + log_pib_pc_mean +
       C(ano)""",
    data=df_model2
).fit(method="bfgs", maxiter=500)

Optimization terminated successfully.
         Current function value: 0.245592
         Iterations: 135
         Function evaluations: 139
         Gradient evaluations: 139


In [ ]:
modelo2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               Logit Regression Results                              
=====================================================================================
Dep. Variable:     gestao_fiscal_responsavel   No. Observations:                  135
Model:                                 Logit   Df Residuals:                      121
Method:                                  MLE   Df Model:                           13
Date:                       Mon, 09 Mar 2026   Pseudo R-squ.:                  0.3252
Time:                               06:04:30   Log-Likelihood:                -33.155
converged:                              True   LL-Null:                       -49.135
Covariance Type:                   nonrobust   LLR p-value:                  0.002435
====================================================================================
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept          -37.1085     14.353     -2.585      0.010     -65.240      -8.977
C(ano)[T.2020]       0.0915      0.951      0.096      0.923      -1.773       1.956
C(ano)[T.2021]      -1.5006      1.928     -0.778      0.436      -5.279       2.278
C(ano)[T.2022]      -1.5961      2.714     -0.588      0.556      -6.915       3.723
C(ano)[T.2023]      -4.8009      3.459     -1.388      0.165     -11.580       1.979
austeridade          0.5622      0.823      0.683      0.495      -1.052       2.176
mesmopartido         3.8230      1.405      2.721      0.007       1.070       6.577
reeleito            -1.0727      1.187     -0.904      0.366      -3.399       1.253
log_pib_pc          15.6899      8.553      1.834      0.067      -1.074      32.454
log_pop            -34.4677     30.971     -1.113      0.266     -95.170      26.235
arcsinh_dcl         -1.4974      0.691     -2.167      0.030      -2.852      -0.143
arcsinh_dcl_mean     0.1203      0.075      1.604      0.109      -0.027       0.267
log_pop_mean        35.6873     31.097      1.148      0.251     -25.261      96.636
log_pib_pc_mean    -10.4821      8.393     -1.249      0.212     -26.933       5.968
====================================================================================

Possibly complete quasi-separation: A fraction 0.13 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

C(uf) no Logit binário com painel curto sempre colapsa quando há UFs com Y constante — que é o caso de 74% das UFs aqui. A solução canônica é exatamente o Mundlak: médias intra-UF absorvem a heterogeneidade individual sem criar separação perfeita.

- **Problema 1** — Separação perfeita pelas dummies de UF: 19 UFs têm Y=1 em todos os anos e 1 UF (RN) tem Y=0 em todos. Para essas 20 UFs, a dummy correspondente separa perfeitamente o outcome — o MLE tenta empurrar o coeficiente para ±∞, a Hessiana vira singular e o algoritmo diverge (function value = inf).

- **Problema 2** — Ocorre o chamado "Incidental parameters problem"! Com N=135 e 26 dummies de UF + 4 de ano + 6 regressores = 37 parâmetros, a razão N/k = 3.6 está muito abaixo do mínimo recomendado (~10). Os coeficientes das dummies são estimados com apenas 5 observações cada.

### 3.3 Modelo 3 - Mundlak-Wooldridge ( log_pop e log_pib_pc pelas suas médias dentro do grupo UF)

In [ ]:
# Eliminar log_pop (colinear com log_pop_mean, sem variação within)
# Tirar log_pib_pc se mantiver colinearidade com log_rcl
# Mundlak só para variáveis com variação temporal real

# Criação de replica
df_model3 = df.copy()


for v in ['arcsinh_dcl', 'log_rcl']:
    df_model3[f'{v}_mean'] = df.groupby('uf')[v].transform('mean')

modelo3 = smf.logit(
    """gestao_fiscal_responsavel ~
       austeridade + mesmopartido + reeleito +
       log_rcl + arcsinh_dcl +
       arcsinh_dcl_mean + log_rcl_mean +
       C(ano)""",
    data=df_model3
).fit(method='bfgs', maxiter=500)

Optimization terminated successfully.
         Current function value: 0.287443
         Iterations: 104
         Function evaluations: 108
         Gradient evaluations: 108


In [ ]:
modelo3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               Logit Regression Results                              
=====================================================================================
Dep. Variable:     gestao_fiscal_responsavel   No. Observations:                  135
Model:                                 Logit   Df Residuals:                      123
Method:                                  MLE   Df Model:                           11
Date:                       Mon, 09 Mar 2026   Pseudo R-squ.:                  0.2102
Time:                               06:06:44   Log-Likelihood:                -38.805
converged:                              True   LL-Null:                       -49.135
Covariance Type:                   nonrobust   LLR p-value:                   0.03707
====================================================================================
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept          -24.7222     12.794     -1.932      0.053     -49.798       0.353
C(ano)[T.2020]      -0.0502      0.975     -0.052      0.959      -1.961       1.861
C(ano)[T.2021]      -0.3515      2.055     -0.171      0.864      -4.379       3.676
C(ano)[T.2022]       0.0996      2.960      0.034      0.973      -5.702       5.901
C(ano)[T.2023]      -1.8116      3.396     -0.533      0.594      -8.468       4.845
austeridade          0.3905      0.726      0.538      0.591      -1.033       1.814
mesmopartido         2.1966      0.981      2.238      0.025       0.273       4.120
reeleito            -0.6619      1.021     -0.648      0.517      -2.664       1.340
log_rcl              4.2294      7.293      0.580      0.562     -10.064      18.523
arcsinh_dcl         -1.0676      0.565     -1.888      0.059      -2.176       0.041
arcsinh_dcl_mean     0.0594      0.056      1.064      0.287      -0.050       0.169
log_rcl_mean        -2.1059      7.429     -0.283      0.777     -16.667      12.456
====================================================================================

Possibly complete quasi-separation: A fraction 0.10 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

**Note**: Resultado na varíável **mesmopartido**.

### 3.4 Modelo 4 - Conditional Logit ((Chamberlain, 1980)

In [ ]:
from statsmodels.discrete.conditional_models import ConditionalLogit
import pandas as pd

# efeitos fixos de ano : ano_2020, ano_2021...
df_model4 = df.copy() # Work on a copy to avoid side effects

# Identifica o grupo de UF onde internamente o Y é constante
constant_y_ufs = df_model4.groupby('uf')['gestao_fiscal_responsavel'].nunique()
ufs_to_exclude = constant_y_ufs[constant_y_ufs == 1].index

# Se a lista de UF não for vazia então elimine elas do dataset
if not ufs_to_exclude.empty:
    df_model4 = df_model4[~df_model4['uf'].isin(ufs_to_exclude)]

# Transforme 'ano' to para um rótulo
df_model4['ano'] = df_model4['ano'].astype(str)
df_model4 = pd.get_dummies(df_model4, columns=["ano"], drop_first=True)

# variáveis explicativas
X = df_model4[[
    "austeridade",
    "mesmopartido",
    "reeleito",
    "log_pib_pc",
    "log_pop",
    "arcsinh_dcl",
    "ano_2020",
    "ano_2021",
    "ano_2022",
    "ano_2023"
]].astype(float)

y = df_model4["gestao_fiscal_responsavel"].astype(int) # Explicitly cast to int

# grupos = estados
groups = df_model4["uf"]

# modelo logit condicional
modelo4 = ConditionalLogit(y, X, groups=groups).fit(method='bfgs', maxiter=2000)

ValueError: need covariance of parameters for computing (unnormalized) covariances

In [ ]:
df_model4[[
"ano_2020",
"ano_2021",
"ano_2022",
"ano_2023"
]].sum()

,0
ano_2020,7
ano_2021,7
ano_2022,7
ano_2023,7


In [ ]:
X.corr()

,austeridade,mesmopartido,reeleito,log_pib_pc,log_pop,arcsinh_dcl,ano_2020,ano_2021,ano_2022,ano_2023
austeridade,1.000000,-0.315447,0.114708,0.738055,0.689231,0.070938,-0.028677,-0.028677,-0.028677,0.114708
mesmopartido,-0.315447,1.000000,0.285714,-0.420053,0.091643,-0.273740,-0.071429,-0.071429,-0.071429,0.285714
reeleito,0.114708,0.285714,1.000000,0.288975,0.008000,-0.106127,-0.250000,-0.250000,-0.250000,1.000000
log_pib_pc,0.738055,-0.420053,0.288975,1.000000,0.323733,-0.196841,-0.191750,0.031263,0.157435,0.288975
log_pop,0.689231,0.091643,0.008000,0.323733,1.000000,0.135190,-0.005113,0.001750,0.008000,0.008000
arcsinh_dcl,0.070938,-0.273740,-0.106127,-0.196841,0.135190,1.000000,0.234418,-0.266079,-0.114195,-0.106127
ano_2020,-0.028677,-0.071429,-0.250000,-0.191750,-0.005113,0.234418,1.000000,-0.250000,-0.250000,-0.250000
ano_2021,-0.028677,-0.071429,-0.250000,0.031263,0.001750,-0.266079,-0.250000,1.000000,-0.250000,-0.250000
ano_2022,-0.028677,-0.071429,-0.250000,0.157435,0.008000,-0.114195,-0.250000,-0.250000,1.000000,-0.250000
ano_2023,0.114708,0.285714,1.000000,0.288975,0.008000,-0.106127,-0.250000,-0.250000,-0.250000,1.000000


**IMPORTANTE**: COlapso do Conditional Logit não é erro de programa. Ano_2023 coimo dummy completamente correlacionada com reeeleito, ou seja, só temos reeleitos em 2023 na base e se temos, é no ano de 2023. NEssa situação deve-se eliminar ou ano_2023 ou  'reeleito'.

In [ ]:
modelo4.summary()

## Conclusão

Percebida um quasi-separação parcial em 7 observações com DCL fortemente negativa, assim corresponde nesses casos a  todas com Y=1..

O principal resultado está no **modelo 3**  é que vislumbra-se que  **mesmopartido** está significativo (p=0.025, OR≈9) — não é afetado pela quasi-separação de forma alguma, pois vem de  variação independente localizada apenas na dimensão político, por isso não no eixo da DCL!

O estimador de Logit Condicional (Chamberlain, 1980) não é identificado neste painel devido à separação quasi-perfeita (dentro da UF). Por exemplo,  em Mato Grosso e Paraíba, onde a transição da DCL de positiva para negativa coincide com a mudança em Y. O estimador de Mundlak-Wooldridge (1978) é adotado como alternativa, controlando a correlação entre os efeitos individuais e os regressores mediante a inclusão das médias (dentro da UF)

Logit Condicional (Chamberlain, 1980) estima os coeficientes usando apenas a variação within-UF,ou seja, dentro da propria UF se ela em algum momento transitou de Y=0 pra Y=1 ou Y=1 foi para Y=0.. Para isso funcionar, as variáveis explicativas precisam ter variação within que não separe Y perfeitamente.



---
## 7. Referências


- BRASIL. Lei Complementar nº 101, de 4 de maio de 2000. Estabelece normas de finanças públicas voltadas para a responsabilidade na gestão fiscal e dá outras providências. Diário Oficial da União: seção 1, Brasília, DF, ano 138, n. 86-E, p. 1-17, 5 maio 2000. Disponível em: planalto.gov.br. Acesso em: 1 mar. 2026.

- CHAMBERLAIN, Gary. Analysis of Covariance with Qualitative Data. The Review of Economic Studies, [Oxford], v. 47, n. 1, p. 225-238, jan. 1980.

- MUNDLAK, Yair. On the Pooling of Time Series and Cross Section Data. Econometrica, v. 46, n. 1, p. 69-85, jan. 1978. Disponível em: https://www.jstor.org/stable/1913646. Acesso em: 9 mar. 2026.

